In [1]:
!pip install -U pypdf langchain_community chromadb langchain langchain_openai openai tiktoken rank_bm25 sentence_transformers cohere langchain_cohere flashrank faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.9/643.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.6/340.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.2/259.2 kB 14.7 MB/s eta 0:0

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.docstore.document import Document
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_cohere import CohereRerank
import cohere
from langchain.document_loaders import PyPDFLoader
from google.colab import drive
from langchain.vectorstores import Chroma
import chromadb
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import langchain
from langchain_community.vectorstores import FAISS

In [3]:
OPENAI_API_TOKEN=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_TOKEN

In [4]:
documents = TextLoader("/content/state_of_the_union.txt").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [5]:
embeddings = OpenAIEmbeddings()

In [6]:
retriever = FAISS.from_documents(texts, embeddings).as_retriever(search_kwargs={"k": 10})

In [7]:
query = "What did the president say about Ketanji Brown Jackson"

In [8]:
docs = retriever.invoke(query)

In [9]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

In [10]:
pretty_print_docs(docs)

Document 1:

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.
Metadata: {'source': '/content/state_of_the_union.txt'}
----------------------------------------------------------------------------------------------------
Document 2:

As I said last year, especially to our younger transgender Americans, I will always have your back as your President, so you can be yourself and reach your God-given potential. 

While it often appears that we never agree, that isn’t true. I signed 80 bipartisan bills into law last year. From preventing government shutdowns to protecting Asian-Americans from still-too-common hate crimes to reforming military justice.
Metadata: {'source': '/content/state_of_the_union.txt'}
-------

In [11]:
llm = ChatOpenAI()

In [15]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers.document_compressors import EmbeddingsFilter

In [13]:
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever=ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)
compressed_docs = compression_retriever.invoke("What did the president say about Ketanji Jackson Brown")
compressed_docs

[Document(metadata={'source': '/content/state_of_the_union.txt'}, page_content='When I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.')]

In [14]:
compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")
pretty_print_docs(compressed_docs)

Document 1:

- More infrastructure and innovation in America.
- More goods moving faster and cheaper in America.
- More jobs where you can earn a good living in America.
Metadata: {'source': '/content/state_of_the_union.txt'}
----------------------------------------------------------------------------------------------------
Document 2:

Ban assault weapons and high-capacity magazines.
Metadata: {'source': '/content/state_of_the_union.txt'}


In [18]:
embeddings_filter = EmbeddingsFilter(embeddings=embeddings)
compression_retriever3 = ContextualCompressionRetriever(base_compressor=embeddings_filter, base_retriever=retriever)
compressed_docs4 = compression_retriever3.invoke("What were the top three priorities outlined in the most recent State of the Union address?")
pretty_print_docs(compressed_docs4)

Document 1:

The only nation that can be defined by a single word: possibilities. 

So on this night, in our 245th year as a nation, I have come to report on the State of the Union. 

And my report is this: the State of the Union is strong—because you, the American people, are strong. 

We are stronger today than we were a year ago. 

And we will be stronger a year from now than we are today. 

Now is our moment to meet and overcome the challenges of our time. 

And we will, as one people. 

One America.
Metadata: {'source': '/content/state_of_the_union.txt'}
----------------------------------------------------------------------------------------------------
Document 2:

And soon, we’ll strengthen the Violence Against Women Act that I first wrote three decades ago. It is important for us to show the nation that we can come together and do big things. 

So tonight I’m offering a Unity Agenda for the Nation. Four big things we can do together.  

First, beat the opioid epidemic. 

There 

Using Compression Retriever

In [19]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [20]:
query="What were the top three priorities outlined in the most recent State of the Union address?"

In [21]:
chain.invoke(query)

{'query': 'What were the top three priorities outlined in the most recent State of the Union address?',
 'result': 'The top three priorities outlined in the most recent State of the Union address were supporting working families, investing in healthcare and education, and supporting veterans.'}

In [22]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter

In [26]:
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0, separator=". ")
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)

In [27]:
pipeline_compressor = DocumentCompressorPipeline(transformers=[splitter, redundant_filter, relevant_filter]) #Combine all filters using DocCompressor

Using a combinatin of multiple doc filters

In [29]:
compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline_compressor, base_retriever=retriever)

In [30]:
compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")
pretty_print_docs(compressed_docs)

Document 1:

The only nation that can be defined by a single word: possibilities. 

So on this night, in our 245th year as a nation, I have come to report on the State of the Union. 

And my report is this: the State of the Union is strong—because you, the American people, are strong
Metadata: {'source': '/content/state_of_the_union.txt'}
----------------------------------------------------------------------------------------------------
Document 2:

But that trickle-down theory led to weaker economic growth, lower wages, bigger deficits, and the widest gap between those at the top and everyone else in nearly a century. 

Vice President Harris and I ran for office with a new economic vision for America. 

Invest in America. Educate Americans
Metadata: {'source': '/content/state_of_the_union.txt'}
----------------------------------------------------------------------------------------------------
Document 3:

And let’s get all Americans the mental health services they need. More people 

In [31]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [32]:
query="What were the top three priorities outlined in the most recent State of the Union address?"

In [33]:
chain.invoke(query)

{'query': 'What were the top three priorities outlined in the most recent State of the Union address?',
 'result': 'The top three priorities outlined in the most recent State of the Union address were:\n1. Passing the Bipartisan Infrastructure Law\n2. Providing a pathway to citizenship for Dreamers, essential workers, and farmworkers\n3. Addressing the issue of inflation and working to lower costs and the deficit.'}